In [46]:
# Load all data files
import json
from pathlib import Path
import pandas as pd

In [47]:
print("Loading data files...")

# 1. Load mapping file (old_id -> new_id)
mapping_dict = {}
with open('mapeamento.txt', 'r') as f:
    for line in f:
        if '->' in line:
            old_path, new_path = line.strip().split(' -> ')
            # Extract just the filename without .mp3
            old_id = Path(old_path).stem
            new_id = Path(new_path).stem
            mapping_dict[old_id] = new_id

print(f"Loaded {len(mapping_dict)} mappings")

# 2. Load labels (new_id -> "ia" or "human")
with open('labels.json', 'r') as f:
    labels_dict = json.load(f)

print(f"Loaded {len(labels_dict)} labels")

# 3. Load lyrics info for similar pairs
with open('similares_hasLyrics.json', 'r') as f:
    similares_lyrics = json.load(f)

# 4. Load lyrics info for random pairs  
with open('random_hasLyrics.json', 'r') as f:
    random_lyrics = json.load(f)

# Combine lyrics info
lyrics_dict = {**similares_lyrics, **random_lyrics}
print(f"Loaded lyrics info for {len(lyrics_dict)} tracks")

# 5. Load Suno AI dataset
suno_df = pd.read_csv('suno_ai_dataset_with_lyrics_20250515_204541.csv')
print(f"Loaded Suno dataset: {len(suno_df)} tracks")

# 6. Load MTG-Jamendo dataset
jamendo_df = pd.read_csv('mtg_jamendo_with_genres.csv')
print(f"Loaded Jamendo dataset: {len(jamendo_df)} tracks")

print("\nAll data loaded successfully!")

Loading data files...
Loaded 30 mappings
Loaded 30 labels
Loaded lyrics info for 30 tracks
Loaded Suno dataset: 4059 tracks
Loaded Jamendo dataset: 55701 tracks

All data loaded successfully!


In [48]:
# Load additional Suno AI data from JSONL file
print("Loading Suno AI posts from JSONL...")

# Load the JSONL file
suno_posts = []
jsonl_path = './suno_ai_posts_with_genres.jsonl'

try:
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                post_data = json.loads(line)
                suno_posts.append(post_data)

    print(f"Loaded {len(suno_posts)} posts from JSONL file")

    # Create a lookup dictionary by post ID for easy access
    suno_posts_dict = {post['id']: post for post in suno_posts}

    
except FileNotFoundError:
    print(f"JSONL file not found at {jsonl_path}")
    suno_posts_dict = {}
except Exception as e:
    print(f"Error loading JSONL file: {e}")
    suno_posts_dict = {}

Loading Suno AI posts from JSONL...
Loaded 33626 posts from JSONL file
Loaded 33626 posts from JSONL file


In [49]:
# Load genres_musics.json for accurate genre scores
print("Loading genres_musics.json for accurate genre filtering...")

try:
    with open('genres_musics.json', 'r') as f:
        genres_musics_dict = json.load(f)
    print(f"Loaded genre data for {len(genres_musics_dict)} tracks")

except FileNotFoundError:
    print("genres_musics.json not found")
    genres_musics_dict = {}
except Exception as e:
    print(f"Error loading genres_musics.json: {e}")
    genres_musics_dict = {}

Loading genres_musics.json for accurate genre filtering...
Loaded genre data for 30 tracks


In [50]:
# Process AI tracks (using genres_musics.json for consistency)
print("Processing AI tracks...")

ai_tracks = []

# Get all AI track IDs from labels
ai_ids = [track_id for track_id, label in labels_dict.items() if label == "ia"]
print(f"Found {len(ai_ids)} AI tracks")


def filter_genres_from_json(track_data, threshold=0.4):
    """Filter genres based on score threshold using JSON data"""
    if not track_data:
        return ""

    top_3_genres = track_data.get("top_3_genres", [])
    genre_scores = track_data.get("genre_scores", [])

    if len(top_3_genres) != len(genre_scores):
        return ""

    # Filter genres with scores >= threshold
    filtered_genres = []
    for genre, score in zip(top_3_genres, genre_scores):
        if score >= threshold:
            filtered_genres.append(genre)

    return "|".join(filtered_genres)


# Create reverse mapping (new_id -> old_id) for AI tracks
reverse_mapping = {new_id: old_id for old_id, new_id in mapping_dict.items()}

for new_id in ai_ids:
    # Check if we have data in genres_musics.json first
    if new_id in genres_musics_dict:
        track_data = genres_musics_dict[new_id]
        old_id = reverse_mapping.get(new_id)

        # Apply accurate threshold filtering using JSON data
        predicted_genres_filtered = filter_genres_from_json(track_data, threshold=0.4)
        top_3_genres_formatted = "|".join(track_data.get("top_3_genres", []))

        has_lyrics = False

        # Try to get lyrics info from Suno dataset or JSONL
        if old_id:
            suno_row = suno_df[suno_df["id"] == old_id]
            if len(suno_row) > 0:
                has_lyrics = not suno_row.iloc[0]["instrumental"]
            elif old_id in suno_posts_dict:
                has_lyrics = True

        track_data_record = {
            "id": new_id,
            "type": "ai",
            "predicted_genres": predicted_genres_filtered,
            "has_lyrics": has_lyrics,
        }
        ai_tracks.append(track_data_record)
        print(
            f"Processed AI track {new_id} from JSON: predicted_genres='{predicted_genres_filtered}', top_3='{top_3_genres_formatted}'"
        )
    else:
        print(f"Warning: AI track {new_id} not found in genres_musics.json")

print(f"Processed {len(ai_tracks)} AI tracks from genres_musics.json")

# Display sample
ai_df = pd.DataFrame(ai_tracks)
print("\nSample AI tracks:")
print(ai_df.head())

Processing AI tracks...
Found 15 AI tracks
Processed AI track l11vtp2a from JSON: predicted_genres='rock', top_3='rock|pop|poprock'
Processed AI track zn5cz600 from JSON: predicted_genres='classical', top_3='classical|soundtrack|ambient'
Processed AI track 123m509s from JSON: predicted_genres='ambient|electronic', top_3='ambient|electronic|soundtrack'
Processed AI track bzwu2027 from JSON: predicted_genres='chillout|pop', top_3='chillout|pop|downtempo'
Processed AI track 7s3o4ys7 from JSON: predicted_genres='metal|rock', top_3='metal|rock|electronic'
Processed AI track znxd2wka from JSON: predicted_genres='hiphop|rap', top_3='hiphop|rap|electronic'
Processed AI track g1jkyftb from JSON: predicted_genres='pop', top_3='pop|rock|blues'
Processed AI track okch36rd from JSON: predicted_genres='electronic', top_3='electronic|pop|easylistening'
Processed AI track jfxaxuvc from JSON: predicted_genres='rock', top_3='rock|metal|electronic'
Processed AI track 5q92om02 from JSON: predicted_genres=

In [51]:
# Process Human tracks (using genres_musics.json for accurate filtering)
print("Processing Human tracks...")

human_tracks = []

# Get all human track IDs from labels
human_ids = [track_id for track_id, label in labels_dict.items() if label == "human"]
print(f"Found {len(human_ids)} human tracks")


def filter_genres_from_json(track_data, threshold=0.4):
    """Filter genres based on score threshold using JSON data"""
    if not track_data:
        return ""

    top_3_genres = track_data.get("top_3_genres", [])
    genre_scores = track_data.get("genre_scores", [])

    if len(top_3_genres) != len(genre_scores):
        return ""

    # Filter genres with scores > threshold
    filtered_genres = []
    for genre, score in zip(top_3_genres, genre_scores):
        if score > threshold:
            filtered_genres.append(genre)

    return "|".join(filtered_genres)


for new_id in human_ids:
    # Check if we have data in genres_musics.json
    if new_id in genres_musics_dict:
        track_data = genres_musics_dict[new_id]

        # Get lyrics info
        has_lyrics = (
            lyrics_dict.get(new_id, 0) == 1
        )  # 1 means has lyrics, 0 means no lyrics

        # Use the old_id from the mapping for consistency
        reverse_mapping = {new_id: old_id for old_id, new_id in mapping_dict.items()}
        old_id = reverse_mapping.get(new_id)

        # Apply accurate threshold filtering using JSON data
        predicted_genres_filtered = filter_genres_from_json(track_data, threshold=0.4)
        top_3_genres_formatted = "|".join(track_data.get("top_3_genres", []))

        # Extract data according to requirements
        track_data_record = {
            "id": new_id,
            "type": "human",
            "predicted_genres": predicted_genres_filtered,
            "has_lyrics": has_lyrics,
        }
        human_tracks.append(track_data_record)
        print(
            f"Processed human track {new_id} from JSON: predicted_genres='{predicted_genres_filtered}', top_3='{top_3_genres_formatted}'"
        )
    else:
        print(f"Warning: Human track {new_id} not found in genres_musics.json")

print(f"Processed {len(human_tracks)} human tracks from genres_musics.json")

# Display sample
human_df = pd.DataFrame(human_tracks)
print("\nSample human tracks:")
print(human_df.head())

Processing Human tracks...
Found 15 human tracks
Processed human track 6sdqefhc from JSON: predicted_genres='rock', top_3='rock|indie|poprock'
Processed human track uc35a8er from JSON: predicted_genres='classical', top_3='classical|orchestral|soundtrack'
Processed human track 6a74vba1 from JSON: predicted_genres='ambient', top_3='ambient|electronic|soundtrack'
Processed human track 0pce8u6v from JSON: predicted_genres='pop', top_3='pop|electronic|rock'
Processed human track 3kuhna98 from JSON: predicted_genres='metal', top_3='metal|electronic|industrial'
Processed human track 2dyp2m8k from JSON: predicted_genres='hiphop', top_3='hiphop|rap|pop'
Processed human track oiiosevh from JSON: predicted_genres='pop', top_3='pop|popfolk|poprock'
Processed human track mu0n03zr from JSON: predicted_genres='electronic', top_3='electronic|ambient|chillout'
Processed human track vc0afghx from JSON: predicted_genres='hiphop|rap', top_3='hiphop|rap|electronic'
Processed human track xxzmx3db from JSON:

In [52]:
# Combine all tracks and create final dataset
print("Combining all tracks...")

# Now AI tracks and human tracks have the same format, so we can combine them directly
all_tracks = human_tracks + ai_tracks

# Create final DataFrame
final_df = pd.DataFrame(all_tracks)

print(f"Final dataset contains {len(final_df)} tracks:")
print(f"- Human tracks: {len(human_tracks)}")
print(f"- AI tracks: {len(ai_tracks)}")

# Display summary statistics
print("\nDataset summary:")
print(final_df['type'].value_counts())
print("\nHas lyrics distribution:")
print(final_df['has_lyrics'].value_counts())

print("\nSample of final dataset:")
print(final_df.head(10))

Combining all tracks...
Final dataset contains 30 tracks:
- Human tracks: 15
- AI tracks: 15

Dataset summary:
type
human    15
ai       15
Name: count, dtype: int64

Has lyrics distribution:
has_lyrics
True     23
False     7
Name: count, dtype: int64

Sample of final dataset:
         id   type predicted_genres  has_lyrics
0  6sdqefhc  human             rock        True
1  uc35a8er  human        classical        True
2  6a74vba1  human          ambient       False
3  0pce8u6v  human              pop        True
4  3kuhna98  human            metal       False
5  2dyp2m8k  human           hiphop        True
6  oiiosevh  human              pop        True
7  mu0n03zr  human       electronic       False
8  vc0afghx  human       hiphop|rap        True
9  xxzmx3db  human             rock        True


In [53]:
# Export the final dataset
output_filename = 'combined_dataset_with_lyrics.csv'
final_df.to_csv(output_filename, index=False)

print(f"Dataset exported to: {output_filename}")
print(f"Total tracks exported: {len(final_df)}")

# Show detailed breakdown
print("\nDetailed breakdown:")
breakdown = final_df.groupby(['type', 'has_lyrics']).size().reset_index(name='count')
print(breakdown)

# Show some examples of each type/
print("\n=== AI Tracks Examples ===")
ai_examples = final_df[final_df['type'] == 'ai'].head(3)
for idx, row in ai_examples.iterrows():
    print(f"ID: {row['id']}, Has Lyrics: {row['has_lyrics']}, Genres: {row['predicted_genres']}")

print("\n=== Human Tracks Examples ===")
human_examples = final_df[final_df['type'] == 'human'].head(3)
for idx, row in human_examples.iterrows():
    print(f"New ID: {row['id']}, Has Lyrics: {row['has_lyrics']}, Genres: {row['predicted_genres']}")

print(f"\nFinal dataset saved as '{output_filename}' with {len(final_df)} tracks total.")

Dataset exported to: combined_dataset_with_lyrics.csv
Total tracks exported: 30

Detailed breakdown:
    type  has_lyrics  count
0     ai       False      2
1     ai        True     13
2  human       False      5
3  human        True     10

=== AI Tracks Examples ===
ID: l11vtp2a, Has Lyrics: True, Genres: rock
ID: zn5cz600, Has Lyrics: True, Genres: classical
ID: 123m509s, Has Lyrics: True, Genres: ambient|electronic

=== Human Tracks Examples ===
New ID: 6sdqefhc, Has Lyrics: True, Genres: rock
New ID: uc35a8er, Has Lyrics: True, Genres: classical
New ID: 6a74vba1, Has Lyrics: False, Genres: ambient

Final dataset saved as 'combined_dataset_with_lyrics.csv' with 30 tracks total.
